<a href="https://colab.research.google.com/github/nasa/ASDC_Data_and_User_Services/blob/Effect_of_Lightning_on_tropNO2/TEMPO_GEE_UWG_ASDC_V2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ TITAN — TEMPO Interactive Tool for Atmospheric NO₂

TEMPO Interactive Tool for Atmospheric NO2

| Field | Detail |
|---|---|
| Mission | TEMPO |
| Variable | NO2 Tropospheric Vertical Column |
| Coverage | North America — USA, Canada, Mexico |
| Temporal | Hourly, April 2023 to present |
| Resolution | 11 x 30 km |
| Data Source | NASA/TEMPO/NO2_L3_QA via Google Earth Engine |

## How to Use
1. Install libraries
2. Authenticate Earth Engine
3. Launch TITAN app

## Important Rules
- Run cells TOP TO BOTTOM — never skip
- Use YOUR OWN Google Cloud Project ID
- If you restart the runtime, re-run from Cell 2
- Wait for the checkmark in each cell before running the next

In [ ]:

!pip install earthengine-api \
             geemap \
             matplotlib \
             pandas \
             numpy \
             ipywidgets \
             pytz \
             --quiet

print("=" * 50)
print("  TITAN — Library Installation Complete")
print("=" * 50)
print("")
print("  earthengine-api  installed")
print("  geemap           installed")
print("  matplotlib       installed")
print("  pandas           installed")
print("  numpy            installed")
print("  ipywidgets       installed")
print("  pytz             installed")
print("")
print("  Now run Cell 3")
print("=" * 50)

  TITAN — Library Installation Complete

  earthengine-api  installed
  geemap           installed
  matplotlib       installed
  pandas           installed
  numpy            installed
  ipywidgets       installed
  pytz             installed

  Now run Cell 3


In [ ]:
# ============================================================
#  Google Cloud Project Manager for TITAN
# Automatically lists your existing projects OR creates new one
# ============================================================

import subprocess
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# STEP 1 — Authenticate with Google Cloud (gcloud)
# ============================================================

print("=" * 55)
print("  TITAN — Google Cloud Project Manager")
print("=" * 55)
print("")
print("  Step 1/4: Authenticating with Google Cloud...")
print("")

# Authenticate gcloud — this shares the same login as EE
auth_result = subprocess.run(
    ['gcloud', 'auth', 'application-default', 'login',
     '--quiet', '--no-launch-browser'],
    capture_output=True, text=True
)

# Also authenticate gcloud CLI itself
auth_result2 = subprocess.run(
    ['gcloud', 'auth', 'login',
     '--quiet', '--no-launch-browser'],
    capture_output=True, text=True
)

print("  Step 2/4: Fetching your Google Cloud projects...")
print("")

# ============================================================
# STEP 2 — Fetch All Existing Projects
# ============================================================

def get_all_projects():
    """
    Fetch all GCP projects accessible to the current account.
    Returns list of dicts: {name, projectId, projectNumber,
                             lifecycleState}
    """
    try:
        result = subprocess.run(
            ['gcloud', 'projects', 'list',
             '--format=json', '--quiet'],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0 and result.stdout.strip():
            projects = json.loads(result.stdout)
            return projects
        else:
            return []
    except Exception as e:
        print(f"  Warning: Could not fetch projects: {e}")
        return []


def check_ee_api_enabled(project_id):
    """Check if Earth Engine API is enabled for a project."""
    try:
        result = subprocess.run(
            ['gcloud', 'services', 'list',
             '--project', project_id,
             '--filter', 'earthengine.googleapis.com',
             '--format=json', '--quiet'],
            capture_output=True, text=True, timeout=15
        )
        if result.returncode == 0:
            services = json.loads(result.stdout
                                  if result.stdout.strip()
                                  else '[]')
            return len(services) > 0
        return False
    except Exception:
        return False


def enable_ee_api(project_id, status_html):
    """Enable Earth Engine API for a project."""
    try:
        status_html.value = (
            '<div style="color:#ffaa00;font-size:12px;'
            'padding:6px 10px;">'
            f'Enabling Earth Engine API for '
            f'<b>{project_id}</b>...'
            '<br>This may take 1-2 minutes.'
            '</div>'
        )
        result = subprocess.run(
            ['gcloud', 'services', 'enable',
             'earthengine.googleapis.com',
             '--project', project_id, '--quiet'],
            capture_output=True, text=True, timeout=120
        )
        return result.returncode == 0
    except Exception:
        return False


def create_new_project(project_id, project_name,
                       status_html):
    """Create a new GCP project."""
    try:
        status_html.value = (
            '<div style="color:#ffaa00;font-size:12px;'
            'padding:6px 10px;">'
            f'Creating project <b>{project_id}</b>...'
            '</div>'
        )
        result = subprocess.run(
            ['gcloud', 'projects', 'create',
             project_id,
             '--name', project_name,
             '--quiet'],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0:
            return True, None
        else:
            err = result.stderr.strip()
            return False, err
    except Exception as e:
        return False, str(e)


# ============================================================
# STEP 3 — Build Interactive UI
# ============================================================

projects     = get_all_projects()
active_proj  = [p for p in projects
                if p.get('lifecycleState') == 'ACTIVE']

print(f"  Step 3/4: Found {len(active_proj)} active project(s)")
print("")
print("  Step 4/4: Building project selector UI...")
print("")

# ── Output area ───────────────────────────────────────────────
status_out = widgets.HTML(value='')
result_out = widgets.Output()

# ── Final selected project store ──────────────────────────────
selected_project_store = {'id': None}

# ── Tab: Select Existing Project ─────────────────────────────

if active_proj:
    project_options = [
        (f"{p.get('name','Unknown')}  "
         f"[ID: {p.get('projectId','')}]",
         p.get('projectId', ''))
        for p in active_proj
    ]
else:
    project_options = [('No projects found', '')]

w_project_select = widgets.Dropdown(
    options     = project_options,
    description = 'Project:',
    style       = {'description_width': '70px'},
    layout      = widgets.Layout(width='480px')
)

# EE API status indicator
w_ee_status = widgets.HTML(
    value='<span style="color:gray;font-size:12px;">'
          'Select a project to check EE API status</span>'
)

btn_check_ee = widgets.Button(
    description  = '🔍 Check EE API Status',
    button_style = 'info',
    layout       = widgets.Layout(width='200px', height='32px')
)

btn_enable_ee = widgets.Button(
    description  = '⚡ Enable EE API',
    button_style = 'warning',
    layout       = widgets.Layout(
        width='160px', height='32px',
        visibility='hidden')
)

btn_use_project = widgets.Button(
    description  = '✅ Use This Project',
    button_style = 'success',
    layout       = widgets.Layout(width='180px', height='36px')
)

def on_check_ee(b):
    pid = w_project_select.value
    if not pid:
        w_ee_status.value = (
            '<span style="color:red;font-size:12px;">'
            'No project selected</span>')
        return
    w_ee_status.value = (
        '<span style="color:#ffaa00;font-size:12px;">'
        f'Checking EE API for {pid}...</span>')
    enabled = check_ee_api_enabled(pid)
    if enabled:
        w_ee_status.value = (
            '<span style="color:#00ff99;font-size:12px;">'
            f'Earth Engine API is ENABLED for {pid}</span>')
        btn_enable_ee.layout.visibility = 'hidden'
    else:
        w_ee_status.value = (
            '<span style="color:#ff6b6b;font-size:12px;">'
            f'Earth Engine API is NOT enabled for {pid}  '
            f'- click Enable EE API to fix</span>')
        btn_enable_ee.layout.visibility = 'visible'

btn_check_ee.on_click(on_check_ee)

def on_enable_ee(b):
    pid     = w_project_select.value
    success = enable_ee_api(pid, w_ee_status)
    if success:
        w_ee_status.value = (
            '<span style="color:#00ff99;font-size:12px;">'
            f'Earth Engine API successfully enabled '
            f'for {pid}</span>')
        btn_enable_ee.layout.visibility = 'hidden'
    else:
        w_ee_status.value = (
            '<span style="color:#ff6b6b;font-size:12px;">'
            f'Could not enable EE API automatically.  '
            f'Go to: console.cloud.google.com/apis/library'
            f'?project={pid}</span>')

btn_enable_ee.on_click(on_enable_ee)

def on_use_project(b):
    pid = w_project_select.value
    if not pid:
        status_out.value = (
            '<div style="color:red;font-size:12px;'
            'padding:6px 10px;">'
            'No project selected</div>')
        return
    selected_project_store['id'] = pid
    with result_out:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="background:#0a1a0a;'
            'border:2px solid #00ff99;'
            'border-radius:8px;padding:12px 16px;'
            'font-size:13px;color:#e0ffe0;'
            'line-height:1.9;">'
            '<b style="font-size:15px;color:#00ff99;">'
            'Project Selected!</b><br><br>'
            f'<b>Project ID:</b>  '
            f'<code style="color:#00ccff;">{pid}</code><br>'
            '<br>'
            '<b style="color:#ffaa00;">ACTION REQUIRED:</b><br>'
            'Copy the line below and paste it into '
            '<b>Cell 4</b>:<br><br>'
            '<div style="background:#001a00;'
            'border:1px solid #00ff99;'
            'border-radius:6px;padding:8px 12px;'
            'font-family:monospace;font-size:13px;'
            'color:#00ff99;letter-spacing:0.5px;">'
            f"MY_PROJECT_ID = '{pid}'"
            '</div>'
            '<br>'
            'Then run <b>Cell 4</b> to initialize '
            'Earth Engine.'
            '</div>'
        ))

btn_use_project.on_click(on_use_project)

select_tab_content = widgets.VBox([
    widgets.HTML(
        '<div style="color:#aaaaaa;font-size:12px;'
        'padding:4px 2px 8px 2px;">'
        'These are all Google Cloud projects '
        'associated with your Google account.'
        '</div>'
    ),
    w_project_select,
    widgets.HBox(
        [btn_check_ee, btn_enable_ee],
        layout=widgets.Layout(gap='8px',
                              margin='6px 0')),
    w_ee_status,
    widgets.HTML(
        '<hr style="border:1px dashed #334;'
        'margin:8px 0;">'),
    btn_use_project,
], layout=widgets.Layout(padding='12px'))


# ── Tab: Create New Project ───────────────────────────────────

w_new_name = widgets.Text(
    value       = 'TITAN TEMPO',
    description = 'Name:',
    style       = {'description_width': '70px'},
    layout      = widgets.Layout(width='320px'),
    placeholder = 'e.g. TITAN TEMPO'
)

w_new_id = widgets.Text(
    value       = 'titan-tempo',
    description = 'ID:',
    style       = {'description_width': '70px'},
    layout      = widgets.Layout(width='320px'),
    placeholder = 'e.g. titan-tempo-123'
)

w_id_hint = widgets.HTML(
    value=(
        '<div style="color:#aaaaaa;font-size:11px;'
        'padding:2px 4px;">'
        'ID rules: lowercase letters, numbers, '
        'hyphens only.  6-30 characters.  '
        'Must be globally unique.'
        '</div>'
    )
)

btn_create = widgets.Button(
    description  = '🚀 Create New Project',
    button_style = 'primary',
    layout       = widgets.Layout(width='210px', height='36px')
)

w_create_status = widgets.HTML(value='')

def on_new_name_change(change):
    """Auto-generate a valid project ID from the name."""
    name   = change['new'].lower()
    new_id = (name.replace(' ', '-')
                  .replace('_', '-'))
    # Keep only valid chars
    new_id = ''.join(c for c in new_id
                     if c.isalnum() or c == '-')
    # Trim to 30 chars
    new_id = new_id[:30].strip('-')
    if new_id:
        w_new_id.value = new_id

w_new_name.observe(on_new_name_change, names='value')

def on_create_project(b):
    pid  = w_new_id.value.strip()
    name = w_new_name.value.strip()

    # Validate
    if not pid:
        w_create_status.value = (
            '<div style="color:red;font-size:12px;">'
            'Project ID cannot be empty</div>')
        return
    if len(pid) < 6 or len(pid) > 30:
        w_create_status.value = (
            '<div style="color:red;font-size:12px;">'
            'Project ID must be 6-30 characters</div>')
        return
    if not all(c.isalnum() or c == '-' for c in pid):
        w_create_status.value = (
            '<div style="color:red;font-size:12px;">'
            'Only lowercase letters, numbers, '
            'and hyphens allowed</div>')
        return

    w_create_status.value = (
        '<div style="color:#ffaa00;font-size:12px;">'
        f'Creating project {pid}...  '
        'Please wait.</div>')

    # Create
    success, err = create_new_project(
        pid, name, w_create_status)

    if success:
        # Enable EE API automatically
        w_create_status.value = (
            '<div style="color:#ffaa00;font-size:12px;">'
            f'Project created!  '
            'Enabling Earth Engine API...</div>')
        ee_ok = enable_ee_api(pid, w_create_status)

        selected_project_store['id'] = pid

        with result_out:
            clear_output(wait=True)
            display(widgets.HTML(
                '<div style="background:#0a1a0a;'
                'border:2px solid #00ff99;'
                'border-radius:8px;padding:12px 16px;'
                'font-size:13px;color:#e0ffe0;'
                'line-height:1.9;">'
                '<b style="font-size:15px;'
                'color:#00ff99;">'
                'Project Created!</b><br><br>'
                f'<b>Name:</b>  {name}<br>'
                f'<b>Project ID:</b>  '
                f'<code style="color:#00ccff;">'
                f'{pid}</code><br>'
                f'<b>EE API:</b>  '
                + ('<span style="color:#00ff99;">'
                   'Enabled</span>'
                   if ee_ok else
                   '<span style="color:#ffaa00;">'
                   'Enable manually at '
                   'console.cloud.google.com/apis'
                   '/library</span>')
                + '<br><br>'
                '<b style="color:#ffaa00;">'
                'ACTION REQUIRED:</b><br>'
                'Copy the line below and paste it '
                'into <b>Cell 4</b>:<br><br>'
                '<div style="background:#001a00;'
                'border:1px solid #00ff99;'
                'border-radius:6px;padding:8px 12px;'
                'font-family:monospace;font-size:13px;'
                'color:#00ff99;letter-spacing:0.5px;">'
                f"MY_PROJECT_ID = '{pid}'"
                '</div>'
                '<br>'
                'Then run <b>Cell 4</b> to '
                'initialize Earth Engine.'
                '</div>'
            ))

        w_create_status.value = (
            '<div style="color:#00ff99;font-size:12px;">'
            f'Project {pid} is ready!</div>')
    else:
        if 'already exists' in str(err).lower():
            w_create_status.value = (
                '<div style="color:#ffaa00;font-size:12px;">'
                f'Project ID "{pid}" already exists.  '
                'Try a different ID or use the '
                '"Select Existing" tab.</div>')
        else:
            w_create_status.value = (
                '<div style="color:red;font-size:12px;">'
                f'Failed to create project.<br>'
                f'Error: {err}<br><br>'
                'Try manually at:<br>'
                'console.cloud.google.com'
                '</div>')

btn_create.on_click(on_create_project)

create_tab_content = widgets.VBox([
    widgets.HTML(
        '<div style="color:#aaaaaa;font-size:12px;'
        'padding:4px 2px 8px 2px;">'
        'Create a brand new Google Cloud Project.  '
        'Earth Engine API will be enabled automatically.'
        '</div>'
    ),
    w_new_name,
    w_new_id,
    w_id_hint,
    widgets.HTML(
        '<hr style="border:1px dashed #334;'
        'margin:8px 0;">'),
    btn_create,
    w_create_status,
], layout=widgets.Layout(padding='12px'))


# ── Assemble Tabs ─────────────────────────────────────────────

tabs = widgets.Tab(
    children=[select_tab_content, create_tab_content],
    layout=widgets.Layout(width='600px')
)
tabs.set_title(0,
    f'  Select Existing  ({len(active_proj)} found)  ')
tabs.set_title(1, '  Create New Project  ')

# ── Full UI ───────────────────────────────────────────────────

full_ui = widgets.VBox([

    widgets.HTML(
        '<div style="'
        'background:linear-gradient('
        '135deg,#001a33,#003d99);'
        'color:white;padding:14px 18px;'
        'border-radius:10px;margin-bottom:8px;">'
        '<b style="font-size:16px;">'
        'TITAN — Google Cloud Project Manager'
        '</b><br>'
        '<span style="font-size:12px;opacity:0.80;">'
        'Select an existing project or create a new one  '
        'then copy the project ID into Cell 4'
        '</span>'
        '</div>'
    ),

    tabs,
    status_out,
    result_out,

    widgets.HTML(
        '<div style="background:#1a1a0a;'
        'border:1px solid #554400;'
        'border-radius:8px;padding:10px 14px;'
        'font-size:11.5px;color:#ccaa55;'
        'margin-top:8px;line-height:1.8;">'
        '<b>After selecting or creating a project:</b><br>'
        '1. Copy the  MY_PROJECT_ID = ...  '
        'line shown in the green box<br>'
        '2. Go to Cell 4<br>'
        '3. Replace  your-project-id-here  '
        'with your actual ID<br>'
        '4. Run Cell 4  then Cell 5  then Cell 6'
        '</div>'
    ),

])

display(full_ui)

# Auto-check EE for first project if list not empty
if active_proj:
    print(f"\n  Found {len(active_proj)} active project(s):")
    for p in active_proj[:5]:
        print(f"    - {p.get('name','?'):<25}  "
              f"ID: {p.get('projectId','?')}")
    if len(active_proj) > 5:
        print(f"    ... and {len(active_proj)-5} more")
    print("")
    print("  Use the UI above to select or create a project")
else:
    print("  No existing projects found")
    print("  Use the 'Create New Project' tab above")

print("")
print("  After selecting: copy MY_PROJECT_ID into Cell 4")
print("=" * 55)

  TITAN — Google Cloud Project Manager

  Step 1/4: Authenticating with Google Cloud...

  Step 2/4: Fetching your Google Cloud projects...

  Step 3/4: Found 2 active project(s)

  Step 4/4: Building project selector UI...




  Found 2 active project(s):
    - ASDC-UWG                   ID: asdc-uwg
    - ASDC                       ID: asdc-491915

  Use the UI above to select or create a project

  After selecting: copy MY_PROJECT_ID into Cell 4


In [ ]:
# ============================================================
# CELL 4 — TITAN Interactive Project Selector & Initializer
# ============================================================

import ee
import subprocess
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Global variable that Cell 6 will read ← KEY CONNECTION ──
TITAN_PROJECT_ID = None   # Will be set when user clicks button

# ============================================================
# Fetch existing projects
# ============================================================

def get_projects():
    try:
        r = subprocess.run(
            ['gcloud', 'projects', 'list',
             '--format=json', '--quiet'],
            capture_output=True, text=True, timeout=30)
        if r.returncode == 0 and r.stdout.strip():
            all_p = json.loads(r.stdout)
            return [p for p in all_p
                    if p.get('lifecycleState') == 'ACTIVE']
        return []
    except Exception:
        return []

projects     = get_projects()
proj_options = (
    [('-- Select a project --', '')] +
    [(f"{p.get('name','')}  |  ID: {p.get('projectId','')}",
      p.get('projectId',''))
     for p in projects]
)

# ============================================================
# Widgets
# ============================================================

style_lbl = {'description_width': '120px'}
lay_input = widgets.Layout(width='420px')

w_mode = widgets.ToggleButtons(
    options    = [
        ('  Use Existing Project  ', 'existing'),
        ('  Create New Project    ', 'create' ),
    ],
    value       = 'existing',
    description = '',
    style       = {
        'button_width'     : '220px',
        'description_width': '0px'
    }
)

w_dropdown = widgets.Dropdown(
    options     = proj_options,
    value       = '',
    description = '📂 Project:',
    style       = style_lbl,
    layout      = lay_input
)

w_new_name = widgets.Text(
    value       = 'TITAN TEMPO',
    description = 'Name:',
    placeholder = 'e.g. TITAN TEMPO',
    style       = style_lbl,
    layout      = lay_input
)

w_new_id = widgets.Text(
    value       = 'titan-tempo',
    description = 'Project ID:',
    placeholder = 'e.g. titan-tempo-123',
    style       = style_lbl,
    layout      = lay_input
)

w_id_rule = widgets.HTML(
    value=(
        '<div style="color:#888;font-size:11px;'
        'padding:2px 4px 6px 126px;">'
        'Lowercase letters, numbers, hyphens only  '
        '·  6-30 chars  ·  globally unique'
        '</div>'
    )
)

def on_name_change(change):
    raw    = change['new'].lower()
    gen_id = ''.join(
        c if (c.isalnum() or c == '-') else '-'
        for c in raw.replace(' ', '-')
    ).strip('-')[:30]
    w_new_id.value = gen_id

w_new_name.observe(on_name_change, names='value')

btn_go = widgets.Button(
    description  = '🚀  Initialize TITAN',
    button_style = 'success',
    layout       = widgets.Layout(
        width='200px', height='40px')
)

w_banner = widgets.HTML(value='')
w_result = widgets.Output()

existing_panel = widgets.VBox(
    [w_dropdown],
    layout=widgets.Layout(padding='6px 0'))

create_panel = widgets.VBox(
    [w_new_name, w_new_id, w_id_rule],
    layout=widgets.Layout(padding='6px 0'))

def on_mode_change(change):
    if change['new'] == 'existing':
        existing_panel.layout.display = ''
        create_panel.layout.display   = 'none'
        btn_go.description             = '🚀  Initialize TITAN'
        btn_go.button_style            = 'success'
    else:
        existing_panel.layout.display = 'none'
        create_panel.layout.display   = ''
        btn_go.description             = '🏗️  Create & Initialize'
        btn_go.button_style            = 'primary'
    w_banner.value = ''
    with w_result:
        clear_output(wait=True)

w_mode.observe(on_mode_change, names='value')
create_panel.layout.display = 'none'

# ============================================================
# Helpers
# ============================================================

def set_banner(msg, color='#ffaa00', icon='⏳'):
    w_banner.value = (
        f'<div style="background:#111;'
        f'border-left:4px solid {color};'
        f'border-radius:6px;padding:9px 14px;'
        f'font-size:12.5px;color:{color};'
        f'margin:6px 0;line-height:1.7;">'
        f'{icon}&nbsp; {msg}'
        f'</div>'
    )

def show_success(pid):
    # ── Store globally so Cell 6 can read it ── KEY LINE ──
    global TITAN_PROJECT_ID
    TITAN_PROJECT_ID = pid

    with w_result:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="'
            'background:linear-gradient('
            '135deg,#001a00,#003300);'
            'border:2px solid #00cc66;'
            'border-radius:10px;'
            'padding:16px 20px;'
            'font-size:13px;'
            'color:#ccffcc;'
            'line-height:2.0;">'

            '<div style="display:flex;'
            'align-items:center;gap:10px;'
            'margin-bottom:10px;">'
            '<span style="font-size:28px;">✅</span>'
            '<span style="font-size:17px;'
            'font-weight:bold;color:#00ff88;">'
            'TITAN is Ready!</span>'
            '</div>'

            '<table style="border-collapse:collapse;'
            'width:100%;font-size:12.5px;">'
            '<tr>'
            '<td style="color:#88cc88;padding:3px 12px'
            ' 3px 0;white-space:nowrap;">'
            'Project ID</td>'
            f'<td><code style="color:#00ccff;'
            f'font-size:13px;">{pid}</code></td>'
            '</tr>'
            '<tr>'
            '<td style="color:#88cc88;padding:3px 12px'
            ' 3px 0;">Earth Engine</td>'
            '<td style="color:#00ff88;">Initialized</td>'
            '</tr>'
            '<tr>'
            '<td style="color:#88cc88;padding:3px 12px'
            ' 3px 0;">TEMPO Data</td>'
            '<td style="color:#00ff88;">Accessible</td>'
            '</tr>'
            '<tr>'
            '<td style="color:#88cc88;padding:3px 12px'
            ' 3px 0;">Global Variable</td>'
            f'<td style="color:#00ccff;">'
            f'TITAN_PROJECT_ID = '
            f'<b>{pid}</b></td>'
            '</tr>'
            '</table>'

            '<hr style="border:1px solid #1a4a1a;'
            'margin:12px 0;">'

            '<div style="background:#001a00;'
            'border:1px solid #00cc66;'
            'border-radius:6px;'
            'padding:8px 12px;'
            'font-size:12px;color:#aaffaa;">'
            'Now run <b>Cell 5</b> '
            '(Verification) then '
            '<b>Cell 6</b> (TITAN App)<br>'
            'Cell 6 will automatically use '
            f'<code>{pid}</code> — '
            'no changes needed!'
            '</div>'

            '</div>'
        ))


def show_error(pid, msg):
    if '403' in msg or 'PERMISSION_DENIED' in msg:
        title   = '403 - Permission Denied'
        color   = '#ff6b6b'
        details = (
            'Your account cannot access this project.<br>'
            'Fix: Enable Earth Engine API at:<br>'
            f'<a href="https://console.cloud.google.com/'
            f'apis/library?project={pid}" '
            f'target="_blank" '
            f'style="color:#00ccff;">'
            f'console.cloud.google.com/apis/library'
            f'</a>'
        )
    elif 'NOT_FOUND' in msg or 'not found' in msg.lower():
        title   = 'Project Not Found'
        color   = '#ffaa33'
        details = (
            f'Project ID <code>{pid}</code> '
            f'does not exist.<br>'
            'Fix: Check spelling or use Create New tab.'
        )
    elif 'authorize' in msg.lower() or \
         'authenticate' in msg.lower():
        title   = 'Not Authenticated'
        color   = '#ffaa33'
        details = (
            'Session expired.<br>'
            'Fix: Re-run Cell 3 then retry.'
        )
    elif 'disabled' in msg.lower() or 'API' in msg:
        title   = 'Earth Engine API Not Enabled'
        color   = '#ffaa33'
        details = (
            f'Enable at:<br>'
            f'<a href="https://console.cloud.google.com/'
            f'apis/library?project={pid}" '
            f'target="_blank" '
            f'style="color:#00ccff;">'
            f'console.cloud.google.com/apis/library'
            f'</a><br>'
            'Wait 1 minute then retry.'
        )
    else:
        title   = 'Initialization Error'
        color   = '#ff6b6b'
        details = msg[:300]

    with w_result:
        clear_output(wait=True)
        display(widgets.HTML(
            f'<div style="background:#1a0000;'
            f'border:2px solid {color};'
            f'border-radius:10px;padding:16px 20px;'
            f'font-size:12.5px;color:#ffcccc;'
            f'line-height:1.9;">'
            f'<div style="display:flex;align-items:center;'
            f'gap:10px;margin-bottom:10px;">'
            f'<span style="font-size:26px;">❌</span>'
            f'<span style="font-size:15px;font-weight:bold;'
            f'color:{color};">{title}</span>'
            f'</div>'
            f'<b>Project ID:</b>  '
            f'<code style="color:#ffaa33;">{pid}</code>'
            f'<br><br>{details}'
            f'</div>'
        ))


def create_project_and_init(pid, name):
    set_banner(
        f'Creating project <b>{pid}</b>...',
        '#ffaa00', '🏗️')
    r = subprocess.run(
        ['gcloud', 'projects', 'create', pid,
         '--name', name, '--quiet'],
        capture_output=True, text=True, timeout=60)

    if r.returncode != 0:
        err = r.stderr.strip()
        if 'already exists' in err.lower():
            set_banner(
                f'Project <b>{pid}</b> already exists '
                '— using it.',
                '#ffaa00', '⚠️')
        else:
            set_banner(
                f'Could not create project: {err}',
                '#ff6b6b', '❌')
            show_error(pid, err)
            btn_go.disabled    = False
            btn_go.description = '🏗️  Create & Initialize'
            return

    set_banner(
        f'Enabling Earth Engine API for <b>{pid}</b>...',
        '#ffaa00', '⚡')
    subprocess.run(
        ['gcloud', 'services', 'enable',
         'earthengine.googleapis.com',
         '--project', pid, '--quiet'],
        capture_output=True, text=True, timeout=120)

    do_ee_init(pid)


def do_ee_init(pid):
    set_banner(
        f'Connecting Earth Engine — project <b>{pid}</b>...',
        '#ffaa00', '🛰️')
    try:
        ee.Initialize(project=pid)
        set_banner(
            f'Connected to <b>{pid}</b>',
            '#00cc66', '✅')
        show_success(pid)   # ← sets TITAN_PROJECT_ID
    except Exception as e:
        set_banner('Initialization failed', '#ff6b6b', '❌')
        show_error(pid, str(e))
    finally:
        btn_go.disabled    = False
        btn_go.description = (
            '🚀  Initialize TITAN'
            if w_mode.value == 'existing'
            else '🏗️  Create & Initialize'
        )


# ============================================================
# Button callback
# ============================================================

def on_go(b):
    btn_go.disabled    = True
    btn_go.description = '⏳  Please wait...'
    with w_result:
        clear_output(wait=True)

    if w_mode.value == 'existing':
        pid = w_dropdown.value
        if not pid:
            set_banner(
                'Please select a project from the dropdown.',
                '#ffaa00', '⚠️')
            btn_go.disabled    = False
            btn_go.description = '🚀  Initialize TITAN'
            return
        do_ee_init(pid)
    else:
        pid  = w_new_id.value.strip()
        name = w_new_name.value.strip()
        valid = (
            6 <= len(pid) <= 30 and
            all(c.isalnum() or c == '-' for c in pid) and
            pid[0].isalpha()
        )
        if not valid:
            set_banner(
                'Invalid Project ID.  '
                'Use 6-30 lowercase letters, '
                'numbers, hyphens.  '
                'Must start with a letter.',
                '#ff6b6b', '❌')
            btn_go.disabled    = False
            btn_go.description = '🏗️  Create & Initialize'
            return
        create_project_and_init(pid, name)

btn_go.on_click(on_go)

# ============================================================
# Display UI
# ============================================================

ui = widgets.VBox([
    widgets.HTML(
        '<div style="'
        'background:linear-gradient('
        '135deg,#000d1a,#001a4d,#003d99);'
        'color:white;padding:14px 20px;'
        'border-radius:12px;margin-bottom:8px;'
        'box-shadow:0 3px 14px rgba(0,80,255,0.35);">'
        '<div style="display:flex;'
        'align-items:center;gap:12px;">'
        '<span style="font-size:32px;">🛰️</span>'
        '<div>'
        '<b style="font-size:18px;letter-spacing:1px;">'
        'TITAN</b><br>'
        '<span style="font-size:11.5px;opacity:0.80;">'
        'TEMPO Interactive Tool for Atmospheric NO2'
        ' — Project Setup'
        '</span>'
        '</div></div>'
        '</div>'
    ),
    widgets.HTML(
        '<div style="color:#aaaaaa;font-size:12px;'
        'padding:2px 2px 4px 2px;">'
        'Choose how to set up your Google Cloud Project:'
        '</div>'
    ),
    w_mode,
    widgets.HTML(
        '<hr style="border:1px solid #222;margin:6px 0;">'),
    existing_panel,
    create_panel,
    widgets.HTML(
        f'<div style="color:#556677;font-size:11px;'
        f'padding:2px 4px 8px 4px;">'
        f'{"Found " + str(len(projects)) + " active project(s) on your account." if projects else "No projects found — use Create New tab."}'
        f'</div>'
    ),
    btn_go,
    w_banner,
    w_result,
], layout=widgets.Layout(
    padding       = '10px',
    max_width     = '600px',
    border        = '1px solid #1a2a3a',
    border_radius = '12px',
))

display(ui)

In [ ]:
# ============================================================
# TITAN App
# ============================================================

import ee
import geemap
import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import matplotlib.colors   as mcolors
import matplotlib.colorbar as mcolorbar
import matplotlib.ticker   as mticker
import matplotlib.dates    as mdates
import matplotlib.patches  as mpatches
import matplotlib.lines    as mlines
import ipywidgets          as widgets
import pytz
import io, base64, calendar
from datetime        import datetime, date
from IPython.display import display, clear_output

# ============================================================
# 1) Read Project ID
# ============================================================

def get_titan_project():
    """
    Read the project ID set by Cell 4.
    Falls back gracefully if Cell 4 was not run.
    """
    try:
        # TITAN_PROJECT_ID was set by Cell 4
        pid = TITAN_PROJECT_ID
        if pid:
            return pid
        else:
            return None
    except NameError:
        return None

_project = get_titan_project()

if _project:
    print(f"✅ Using project from Cell 4: {_project}")
    project_id = _project
else:
    print("⚠️  TITAN_PROJECT_ID not found.")
    print("   Cell 4 was not run or did not complete.")
    print("   Please run Cell 4 first and click")
    print("   'Initialize TITAN' before running this cell.")
    raise SystemExit(
        "Run Cell 4 first to select a project.")

# Verify EE is still initialized
try:
    ee.data.getAlgorithms()
    print(f"✅ Earth Engine active — project: {project_id}")
except Exception:
    print(f"🔄 Re-initializing Earth Engine...")
    try:
        ee.Initialize(project=project_id)
        print(f"✅ Re-initialized — project: {project_id}")
    except Exception as e:
        print(f"❌ Could not initialize: {e}")
        print("   Re-run Cell 3 then Cell 4, then retry.")
        raise SystemExit("EE initialization failed.")

# ============================================================
# 2) Geographic Lookup Tables
# ============================================================

US_STATES = {
    'Alabama': '01', 'Alaska': '02', 'Arizona': '04',
    'Arkansas': '05', 'California': '06', 'Colorado': '08',
    'Connecticut': '09', 'Delaware': '10',
    'District of Columbia': '11', 'Florida': '12',
    'Georgia': '13',  'Idaho': '16',
    'Illinois': '17', 'Indiana': '18', 'Iowa': '19',
    'Kansas': '20', 'Kentucky': '21', 'Louisiana': '22',
    'Maine': '23', 'Maryland': '24', 'Massachusetts': '25',
    'Michigan': '26', 'Minnesota': '27', 'Mississippi': '28',
    'Missouri': '29', 'Montana': '30', 'Nebraska': '31',
    'Nevada': '32', 'New Hampshire': '33', 'New Jersey': '34',
    'New Mexico': '35', 'New York': '36',
    'North Carolina': '37', 'North Dakota': '38',
    'Ohio': '39', 'Oklahoma': '40', 'Oregon': '41',
    'Pennsylvania': '42', 'Rhode Island': '44',
    'South Carolina': '45', 'South Dakota': '46',
    'Tennessee': '47', 'Texas': '48', 'Utah': '49',
    'Vermont': '50', 'Virginia': '51', 'Washington': '53',
    'West Virginia': '54', 'Wisconsin': '55', 'Wyoming': '56'
}

CANADA_PROVINCES = [
    'Alberta', 'British Columbia', 'Manitoba',
    'New Brunswick', 'Newfoundland and Labrador',
    'Northwest Territories', 'Nova Scotia', 'Nunavut',
    'Ontario', 'Prince Edward Island', 'Quebec',
    'Saskatchewan', 'Yukon'
]

MEXICO_STATES = [
    'Aguascalientes', 'Baja California',
    'Baja California Sur', 'Campeche', 'Chiapas',
    'Chihuahua', 'Coahuila de Zaragoza', 'Colima',
    'Durango', 'Guanajuato', 'Guerrero', 'Hidalgo',
    'Jalisco', 'Mexico', 'Mexico City',
    'Michoacan de Ocampo', 'Morelos', 'Nayarit',
    'Nuevo Leon', 'Oaxaca', 'Puebla',
    'Queretaro de Arteaga', 'Quintana Roo',
    'San Luis Potosi', 'Sinaloa', 'Sonora', 'Tabasco',
    'Tamaulipas', 'Tlaxcala',
    'Veracruz de Ignacio de la Llave',
    'Yucatan', 'Zacatecas'
]

AVAILABLE_YEARS = [2023, 2024, 2025]
MONTH_NAMES     = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

BAND  = 'vertical_column_troposphere'
SCALE = 5000

visParams = {
    'min'    : 0,
    'max'    : 1.5e16,
    'palette': [
        '000080','0000D9','4000FF','8000FF','0080FF',
        '00D9FF','80FFFF','FF8080','D90000','800000'
    ]
}

# ============================================================
# 3) Timezone Lookup Tables
# ============================================================

US_STATE_TIMEZONES = {
    'Alabama': 'America/Chicago',
    'Alaska': 'America/Anchorage',
    'Arizona': 'America/Phoenix',
    'Arkansas': 'America/Chicago',
    'California': 'America/Los_Angeles',
    'Colorado': 'America/Denver',
    'Connecticut': 'America/New_York',
    'Delaware': 'America/New_York',
    'District of Columbia': 'America/New_York',
    'Florida': 'America/New_York',
    'Georgia': 'America/New_York',
    'Hawaii': 'Pacific/Honolulu',
    'Idaho': 'America/Denver',
    'Illinois': 'America/Chicago',
    'Indiana': 'America/Indiana/Indianapolis',
    'Iowa': 'America/Chicago',
    'Kansas': 'America/Chicago',
    'Kentucky': 'America/New_York',
    'Louisiana': 'America/Chicago',
    'Maine': 'America/New_York',
    'Maryland': 'America/New_York',
    'Massachusetts': 'America/New_York',
    'Michigan': 'America/Detroit',
    'Minnesota': 'America/Chicago',
    'Mississippi': 'America/Chicago',
    'Missouri': 'America/Chicago',
    'Montana': 'America/Denver',
    'Nebraska': 'America/Chicago',
    'Nevada': 'America/Los_Angeles',
    'New Hampshire': 'America/New_York',
    'New Jersey': 'America/New_York',
    'New Mexico': 'America/Denver',
    'New York': 'America/New_York',
    'North Carolina': 'America/New_York',
    'North Dakota': 'America/Chicago',
    'Ohio': 'America/New_York',
    'Oklahoma': 'America/Chicago',
    'Oregon': 'America/Los_Angeles',
    'Pennsylvania': 'America/New_York',
    'Rhode Island': 'America/New_York',
    'South Carolina': 'America/New_York',
    'South Dakota': 'America/Chicago',
    'Tennessee': 'America/Chicago',
    'Texas': 'America/Chicago',
    'Utah': 'America/Denver',
    'Vermont': 'America/New_York',
    'Virginia': 'America/New_York',
    'Washington': 'America/Los_Angeles',
    'West Virginia': 'America/New_York',
    'Wisconsin': 'America/Chicago',
    'Wyoming': 'America/Denver',
}

CANADA_TIMEZONES = {
    'Alberta': 'America/Edmonton',
    'British Columbia': 'America/Vancouver',
    'Manitoba': 'America/Winnipeg',
    'New Brunswick': 'America/Moncton',
    'Newfoundland and Labrador': 'America/St_Johns',
    'Northwest Territories': 'America/Yellowknife',
    'Nova Scotia': 'America/Halifax',
    'Nunavut': 'America/Iqaluit',
    'Ontario': 'America/Toronto',
    'Prince Edward Island': 'America/Halifax',
    'Quebec': 'America/Toronto',
    'Saskatchewan': 'America/Regina',
    'Yukon': 'America/Whitehorse',
}

MEXICO_TIMEZONES = {
    'Aguascalientes': 'America/Mexico_City',
    'Baja California': 'America/Tijuana',
    'Baja California Sur': 'America/Mazatlan',
    'Campeche': 'America/Merida',
    'Chiapas': 'America/Mexico_City',
    'Chihuahua': 'America/Chihuahua',
    'Coahuila de Zaragoza': 'America/Monterrey',
    'Colima': 'America/Mexico_City',
    'Durango': 'America/Mazatlan',
    'Guanajuato': 'America/Mexico_City',
    'Guerrero': 'America/Mexico_City',
    'Hidalgo': 'America/Mexico_City',
    'Jalisco': 'America/Mexico_City',
    'Mexico': 'America/Mexico_City',
    'Mexico City': 'America/Mexico_City',
    'Michoacan de Ocampo': 'America/Mexico_City',
    'Morelos': 'America/Mexico_City',
    'Nayarit': 'America/Mazatlan',
    'Nuevo Leon': 'America/Monterrey',
    'Oaxaca': 'America/Mexico_City',
    'Puebla': 'America/Mexico_City',
    'Queretaro de Arteaga': 'America/Mexico_City',
    'Quintana Roo': 'America/Cancun',
    'San Luis Potosi': 'America/Mexico_City',
    'Sinaloa': 'America/Mazatlan',
    'Sonora': 'America/Hermosillo',
    'Tabasco': 'America/Mexico_City',
    'Tamaulipas': 'America/Monterrey',
    'Tlaxcala': 'America/Mexico_City',
    'Veracruz de Ignacio de la Llave': 'America/Mexico_City',
    'Yucatan': 'America/Merida',
    'Zacatecas': 'America/Mexico_City',
}

# ============================================================
# 4) NO₂ Metadata & Color Config
# ============================================================

NO2_META = {
    'variable'  : 'NO₂ Tropospheric Vertical Column',
    'short_name': 'vertical_column_troposphere',
    'units'     : 'molecules / cm²',
    'vmin'      : 0,
    'vmax'      : 1.5e16,
    'palette'   : [
        '000080','0000D9','4000FF','8000FF','0080FF',
        '00D9FF','80FFFF','FF8080','D90000','800000'
    ],
}

COLORBAR_TICKS  = [0, 3e15, 6e15, 9e15, 1.2e16, 1.5e16]
COLORBAR_LABELS = [
    '0','3×10¹⁵','6×10¹⁵','9×10¹⁵','1.2×10¹⁶','1.5×10¹⁶'
]

TEMPO_START_YEAR  = 2023
TEMPO_START_MONTH = 4

SERIES_COLORS = [
    '#00ccff','#ffaa33','#ff6b6b','#69ff47','#ff69b4',
    '#ffd700','#00ff99','#ff4500','#7b68ee','#00ced1',
    '#ff8c00','#adff2f','#dc143c','#00bfff','#ff1493',
    '#32cd32','#ff6347','#9370db','#20b2aa','#ffa07a',
]

SERIES_LINESTYLES = [
    'solid','dashed','dashdot','dotted',
    (0,(3,1,1,1)),(0,(5,2)),(0,(1,1)),
]

# ============================================================
# 5) Helper Functions
# ============================================================

def get_days_in_month(year, month):
    return calendar.monthrange(year, month)[1]


def get_region(country, state_province, county=None):
    if country == '🇺🇸 United States':
        fips = US_STATES.get(state_province, '48')
        if county and county != '— Entire State —':
            return (ee.FeatureCollection('TIGER/2018/Counties')
                    .filter(ee.Filter.eq('NAME', county))
                    .filter(ee.Filter.eq('STATEFP', fips)))
        return (ee.FeatureCollection('TIGER/2018/States')
                .filter(ee.Filter.eq('STATEFP', fips)))
    elif country == '🇨🇦 Canada':
        if county and county != '— Entire Province —':
            return (ee.FeatureCollection('FAO/GAUL/2015/level2')
                    .filter(ee.Filter.eq('ADM0_NAME','Canada'))
                    .filter(ee.Filter.eq('ADM1_NAME',state_province))
                    .filter(ee.Filter.eq('ADM2_NAME',county)))
        return (ee.FeatureCollection('FAO/GAUL/2015/level1')
                .filter(ee.Filter.eq('ADM0_NAME','Canada'))
                .filter(ee.Filter.eq('ADM1_NAME',state_province)))
    else:
        if county and county != '— Entire State —':
            return (ee.FeatureCollection('FAO/GAUL/2015/level2')
                    .filter(ee.Filter.eq('ADM0_NAME','Mexico'))
                    .filter(ee.Filter.eq('ADM1_NAME',state_province))
                    .filter(ee.Filter.eq('ADM2_NAME',county)))
        return (ee.FeatureCollection('FAO/GAUL/2015/level1')
                .filter(ee.Filter.eq('ADM0_NAME','Mexico'))
                .filter(ee.Filter.eq('ADM1_NAME',state_province)))


def fetch_counties_usa(state_name):
    fips = US_STATES.get(state_name, '48')
    fc   = (ee.FeatureCollection('TIGER/2018/Counties')
            .filter(ee.Filter.eq('STATEFP', fips)))
    return sorted(fc.aggregate_array('NAME').getInfo())


def fetch_subdivisions(country, state_province):
    nation = 'Canada' if country == '🇨🇦 Canada' else 'Mexico'
    fc     = (ee.FeatureCollection('FAO/GAUL/2015/level2')
              .filter(ee.Filter.eq('ADM0_NAME', nation))
              .filter(ee.Filter.eq('ADM1_NAME', state_province)))
    return sorted(fc.aggregate_array('ADM2_NAME').getInfo())


def get_timezone(country, state_province):
    if country == '🇺🇸 United States':
        return US_STATE_TIMEZONES.get(state_province, 'UTC')
    elif country == '🇨🇦 Canada':
        return CANADA_TIMEZONES.get(state_province, 'UTC')
    return MEXICO_TIMEZONES.get(
        state_province, 'America/Mexico_City')


def utc_hour_to_local_label(utc_hour, year, month, tz_str):
    utc_dt   = datetime(year, month, 15, utc_hour, 0,
                        tzinfo=pytz.utc)
    local_tz = pytz.timezone(tz_str)
    local_dt = utc_dt.astimezone(local_tz)
    return (f"{utc_hour:02d}:00 UTC  →  "
            f"{local_dt.strftime('%H:%M')} "
            f"{local_dt.strftime('%Z')}")


def get_tz_abbr(year, month, tz_str):
    try:
        utc_dt = datetime(year, month, 15, 12, 0,
                          tzinfo=pytz.utc)
        return utc_dt.astimezone(
            pytz.timezone(tz_str)).strftime('%Z')
    except Exception:
        return 'UTC'


def format_no2(val):
    if val is None:
        return 'No data'
    try:
        coeff, pwr = f'{float(val):.3e}'.split('e')
        sup = str(int(pwr)).translate(
            str.maketrans('0123456789-','⁰¹²³⁴⁵⁶⁷⁸⁹⁻'))
        return f'{coeff} × 10{sup}  mol/cm²'
    except Exception:
        return str(val)


def build_hour_options(year, month, tz_str):
    return [(utc_hour_to_local_label(h, year, month, tz_str), h)
            for h in range(24)]


def sci_fmt(x, _):
    if x == 0:
        return '0'
    exp = int(np.floor(np.log10(abs(x))))
    c   = x / 10**exp
    sup = str(exp).translate(
        str.maketrans('0123456789-','⁰¹²³⁴⁵⁶⁷⁸⁹⁻'))
    return f'{c:.1f}×10{sup}'


# ============================================================
# 6) Colorbar Builder
# ============================================================

def add_colorbar_to_map(Map):
    hex_colors = ['#' + h for h in NO2_META['palette']]
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'NO2_cmap', hex_colors, N=256)
    norm = mcolors.Normalize(
        vmin=NO2_META['vmin'], vmax=NO2_META['vmax'])
    fig, ax = plt.subplots(figsize=(6.2, 0.60))
    fig.patch.set_alpha(0.0)
    ax.set_facecolor('none')
    cb = mcolorbar.ColorbarBase(
        ax, cmap=cmap, norm=norm,
        orientation='horizontal', ticks=COLORBAR_TICKS)
    cb.set_ticklabels(COLORBAR_LABELS)
    cb.ax.tick_params(labelsize=7, colors='white', length=3)
    cb.outline.set_edgecolor('white')
    cb.outline.set_linewidth(0.8)
    cb.set_label(
        f"{NO2_META['variable']}   [{NO2_META['units']}]",
        fontsize=8, color='white', labelpad=4)
    plt.tight_layout(pad=0.15)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=140,
                bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    Map.add_widget(widgets.HTML(
        '<div style="background:rgba(0,0,20,0.68);'
        'border-radius:8px;padding:4px 10px 2px 10px;">'
        f'<img src="data:image/png;base64,{b64}" '
        'style="display:block;width:430px;height:auto;"/>'
        '</div>'), position='bottomright')


# ============================================================
# 7) EE Sampling Utilities
# ============================================================

def ee_sample_region(region_fc, hour):
    """Monthly spatial mean NO₂ over a region."""
    today   = date.today()
    results = []
    geom    = region_fc.geometry()
    for yr in range(TEMPO_START_YEAR, today.year + 1):
        m_start = TEMPO_START_MONTH \
            if yr == TEMPO_START_YEAR else 1
        m_end   = today.month \
            if yr == today.year else 12
        for mo in range(m_start, m_end + 1):
            last_day   = get_days_in_month(yr, mo)
            start_date = f'{yr}-{mo:02d}-01'
            end_date   = f'{yr}-{mo:02d}-{last_day}'
            try:
                col = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
                       .filterDate(start_date, end_date)
                       .filterBounds(geom)
                       .filter(ee.Filter.calendarRange(
                           hour, hour, 'hour'))
                       .select(BAND))
                n = col.size().getInfo()
                val = (col.mean()
                          .reduceRegion(
                              reducer   = ee.Reducer.mean(),
                              geometry  = geom,
                              scale     = SCALE,
                              maxPixels = 1e9,
                              bestEffort= True)
                          .get(BAND)
                          .getInfo()) if n > 0 else None
            except Exception:
                val, n = None, 0
            results.append({
                'date' : f'{yr}-{mo:02d}',
                'year' : yr, 'month': mo,
                'value': val, 'n'   : n,
            })
    return results


def ee_sample_point(lat, lon, hour):
    """Monthly NO₂ at a single point."""
    point   = ee.Geometry.Point([lon, lat])
    today   = date.today()
    results = []
    for yr in range(TEMPO_START_YEAR, today.year + 1):
        m_start = TEMPO_START_MONTH \
            if yr == TEMPO_START_YEAR else 1
        m_end   = today.month \
            if yr == today.year else 12
        for mo in range(m_start, m_end + 1):
            last_day   = get_days_in_month(yr, mo)
            start_date = f'{yr}-{mo:02d}-01'
            end_date   = f'{yr}-{mo:02d}-{last_day}'
            try:
                col = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
                       .filterDate(start_date, end_date)
                       .filter(ee.Filter.calendarRange(
                           hour, hour, 'hour'))
                       .select(BAND))
                n = col.size().getInfo()
                val = (col.mean()
                          .reduceRegion(
                              reducer  = ee.Reducer.mean(),
                              geometry = point.buffer(SCALE),
                              scale    = SCALE,
                              maxPixels= 1e6)
                          .get(BAND)
                          .getInfo()) if n > 0 else None
            except Exception:
                val, n = None, 0
            results.append({
                'date' : f'{yr}-{mo:02d}',
                'year' : yr, 'month': mo,
                'value': val, 'n'   : n,
            })
    return results


def ee_sample_single(lat, lon, hour, year, month):
    """Sample NO₂ at one point for one month/hour."""
    point      = ee.Geometry.Point([lon, lat])
    last_day   = get_days_in_month(year, month)
    start_date = f'{year}-{month:02d}-01'
    end_date   = f'{year}-{month:02d}-{last_day}'
    try:
        col = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
               .filterDate(start_date, end_date)
               .filter(ee.Filter.calendarRange(
                   hour, hour, 'hour'))
               .select(BAND))
        if col.size().getInfo() == 0:
            return None
        return (col.mean()
                   .reduceRegion(
                       reducer  = ee.Reducer.mean(),
                       geometry = point.buffer(SCALE),
                       scale    = SCALE,
                       maxPixels= 1e6)
                   .get(BAND)
                   .getInfo())
    except Exception:
        return None


# ============================================================
# 8) Multi-Series Time Series Plot
# ============================================================
# ============================================================
# 8) Multi-Series Time Series Plot  (COMPLETE)
# ============================================================

def plot_multi_time_series(series_list, title_extra,
                            out_widget):
    """
    Plot multiple NO₂ time series on one dark figure.
    series_list: list of dicts with keys:
        label, results, color, ls
    """
    if not series_list:
        with out_widget:
            clear_output(wait=True)
            display(widgets.HTML(
                '<div style="color:#ffaa00;padding:14px;">'
                '⚠️ No series to plot.</div>'))
        return

    fig, ax = plt.subplots(figsize=(15, 5.5))
    fig.patch.set_facecolor('#080c14')
    ax.set_facecolor('#0d1117')

    legend_handles = []
    all_values     = []

    for s in series_list:
        results = s['results']
        color   = s['color']
        ls      = s['ls']
        label   = s['label']

        valid   = [(r['date'], r['value'])
                   for r in results
                   if r['value'] is not None]
        no_data = [r['date'] for r in results
                   if r['value'] is None]

        if not valid:
            continue

        dates  = [datetime.strptime(d, '%Y-%m')
                  for d, _ in valid]
        values = [v for _, v in valid]
        all_values.extend(values)

        arr      = np.array(values)
        mean_val = float(np.mean(arr))

        # Shaded fill
        ax.fill_between(dates, values,
                        alpha=0.10, color=color, zorder=1)
        # Main line
        ax.plot(dates, values,
                color=color, linewidth=1.8,
                linestyle=ls, zorder=3,
                solid_capstyle='round')
        # Markers
        ax.scatter(dates, values,
                   color='white', edgecolors=color,
                   s=40, zorder=5, linewidths=0.9)
        # Per-series mean
        ax.axhline(mean_val, color=color,
                   linestyle=':', linewidth=1.0,
                   alpha=0.55, zorder=2)
        # Missing month markers
        for nd in no_data:
            ax.axvline(datetime.strptime(nd, '%Y-%m'),
                       color=color, linestyle=':',
                       linewidth=0.6, alpha=0.25)
        # Max annotation
        if values:
            max_v = max(values)
            max_d = dates[values.index(max_v)]
            ax.annotate(
                f'▲{max_v:.2e}',
                xy=(max_d, max_v),
                xytext=(0, 12),
                textcoords='offset points',
                ha='center', fontsize=7,
                color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->',
                                color=color, lw=0.8))

        legend_handles.append(
            mlines.Line2D(
                [], [],
                color=color, linewidth=2,
                linestyle=ls,
                marker='o', markersize=5,
                markerfacecolor='white',
                markeredgecolor=color,
                label=(f'{label}  '
                       f'[n={len(valid)}  '
                       f'mean={mean_val:.2e}]')
            )
        )

    if not all_values:
        with out_widget:
            clear_output(wait=True)
            display(widgets.HTML(
                '<div style="color:#ffaa00;padding:14px;">'
                '⚠️ All series returned no data.</div>'))
        return

    # Grand mean reference line
    grand_mean = float(np.mean(all_values))
    ax.axhline(grand_mean, color='white',
               linestyle='--', linewidth=0.9,
               alpha=0.35, zorder=2)
    legend_handles.append(
        mlines.Line2D(
            [], [], color='white',
            linewidth=1.5, linestyle='--', alpha=0.5,
            label=f'Grand mean:  {grand_mean:.2e}'))

    # ── Axes formatting ───────────────────────────────────────
    ax.xaxis.set_major_locator(
        mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(
        mdates.DateFormatter('%b\n%Y'))
    ax.tick_params(axis='x', colors='#9aabbb',
                   labelsize=7.0, rotation=0, length=4)
    ax.tick_params(axis='y', colors='#9aabbb',
                   labelsize=8.5, length=4)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(sci_fmt))
    ax.set_xlabel('Month / Year',
                  color='#cccccc', fontsize=10, labelpad=7)
    ax.set_ylabel(
        f'NO₂ Column\n[{NO2_META["units"]}]',
        color='#cccccc', fontsize=10, labelpad=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2a3e')
    ax.grid(color='#151f2e', linestyle='--',
            linewidth=0.7, alpha=0.9)
    ax.set_title(
        f'📈  TEMPO NO₂ Multi-Series  |  {title_extra}\n'
        f'Mission: Apr 2023 → present  |  '
        f'{len(series_list)} series plotted',
        color='white', fontsize=9.5,
        pad=11, loc='left')
    ax.legend(
        handles    = legend_handles,
        loc        = 'upper right',
        fontsize   = 8.0,
        facecolor  = '#0d1117',
        edgecolor  = '#1e2a3e',
        labelcolor = 'white',
        framealpha = 0.90,
        ncol       = max(1, len(legend_handles) // 6),
    )

    plt.tight_layout(pad=1.2)
    with out_widget:
        clear_output(wait=True)
        plt.show()


# ============================================================
# 9) Widget Definitions
# ============================================================

style  = {'description_width': '140px'}
layout = widgets.Layout(width='380px')

w_country = widgets.Dropdown(
    options     = ['🇺🇸 United States',
                   '🇨🇦 Canada','🇲🇽 Mexico'],
    value       = '🇺🇸 United States',
    description = '🌎 Country:',
    style=style, layout=layout)

w_state = widgets.Dropdown(
    options     = sorted(US_STATES.keys()),
    value       = 'Texas',
    description = '🗾 State:',
    style=style, layout=layout)

w_county = widgets.Dropdown(
    options     = ['— Entire State —'],
    value       = '— Entire State —',
    description = '📍 County:',
    style=style, layout=layout)

w_year = widgets.Dropdown(
    options     = AVAILABLE_YEARS,
    value       = 2024,
    description = '📅 Year:',
    style=style, layout=layout)

w_month = widgets.Dropdown(
    options     = [(n, i+1) for i,n in enumerate(MONTH_NAMES)],
    value       = 1,
    description = '🗓️ Month:',
    style=style, layout=layout)

w_basemap = widgets.Dropdown(
    options=[
        ('🛰️  Google Satellite  (True Color)','SATELLITE'),
        ('🌍  Esri World Imagery (True Color)','Esri.WorldImagery'),
    ],
    value       = 'SATELLITE',
    description = '🗺️ Basemap:',
    style=style, layout=layout)

w_region_hour = widgets.Dropdown(
    options     = [(f'{h:02d}:00 UTC', h) for h in range(24)],
    value       = 17,
    description = '🕐 UTC Hour:',
    style       = {'description_width': '90px'},
    layout      = widgets.Layout(width='320px'))

w_region_status = widgets.HTML(
    value=(
        '<span style="color:gray;font-size:12px;">'
        '👆 Click <b>Update Map</b> first, then select '
        'hour and click <b>Plot Regional TS</b></span>'))

region_ts_output = widgets.Output()

coord_style  = {'description_width': '50px'}
coord_layout = widgets.Layout(width='200px')

w_lat = widgets.FloatText(
    value=29.7604, description='📍 Lat:',
    step=0.0001, style=coord_style, layout=coord_layout)
w_lon = widgets.FloatText(
    value=-95.3698, description='📍 Lon:',
    step=0.0001, style=coord_style, layout=coord_layout)

w_ts_hour = widgets.Dropdown(
    options     = [(f'{h:02d}:00 UTC', h) for h in range(24)],
    value       = 17,
    description = '🕐 TS Hour:',
    style       = {'description_width': '80px'},
    layout      = widgets.Layout(width='220px'))

btn_region_ts = widgets.Button(
    description  = '🗺️ Plot Regional Time Series',
    button_style = 'warning',
    layout       = widgets.Layout(width='270px', height='36px'))

btn_sample = widgets.Button(
    description  = '📍 Sample & Plot Point Time Series',
    button_style = 'info',
    layout       = widgets.Layout(width='290px', height='36px'))

btn_update = widgets.Button(
    description  = '🔄  Update Map',
    button_style = 'success',
    layout       = widgets.Layout(width='180px', height='36px'))

w_status = widgets.HTML(
    value=(
        '<span style="color:gray;font-size:13px;">'
        '👆 Select a region and click '
        '<b>Update Map</b></span>'))

w_sample_status = widgets.HTML(
    value=(
        '<span style="color:gray;font-size:12px;">'
        '👆 Enter coordinates and click '
        '<b>Sample &amp; Plot Point Time Series</b></span>'))

w_progress = widgets.IntProgress(
    value=0, min=0, max=24,
    description='Loading:',
    bar_style='info',
    layout=widgets.Layout(width='380px',
                          visibility='hidden'))

map_output    = widgets.Output()
sample_output = widgets.Output()
ts_output     = widgets.Output()

# ============================================================
# 10) Multi-Series Queue Widgets
# ============================================================

_region_queue   = []
queue_output    = widgets.Output()
multi_ts_output = widgets.Output()

btn_add_to_queue = widgets.Button(
    description  = '➕ Add Region + Hour to Queue',
    button_style = 'primary',
    layout       = widgets.Layout(width='260px', height='34px'),
    tooltip      = 'Add current region + hour to queue')

btn_clear_queue = widgets.Button(
    description  = '🗑️ Clear Queue',
    button_style = 'danger',
    layout       = widgets.Layout(width='140px', height='34px'))

btn_plot_queue = widgets.Button(
    description  = '📊 Plot All Queued Series',
    button_style = 'warning',
    layout       = widgets.Layout(width='240px', height='34px'),
    tooltip      = 'Fetch & overlay all queued series')

w_queue_status = widgets.HTML(
    value=(
        '<span style="color:gray;font-size:12px;">'
        '📭 Queue empty — add items above</span>'))

# ============================================================
# 11) App State Store
# ============================================================

_current_region = {
    'fc'    : None,
    'label' : '',
    'tz_str': 'America/Chicago',
}

# ============================================================
# 12) Queue Display Helper
# ============================================================

def refresh_queue_display():
    """Render queue as styled HTML table."""
    with queue_output:
        clear_output(wait=True)
        if not _region_queue:
            display(widgets.HTML(
                '<div style="color:#667788;'
                'font-size:12px;padding:6px 10px;">'
                '📭 Queue is empty.  '
                'Add regions/hours using the buttons above.'
                '</div>'))
            return

        ls_names = [
            'solid','dashed','dashdot','dotted',
            'loosely dashdot','loosely dashed',
            'densely dotted']
        rows = ''
        for i, item in enumerate(_region_queue):
            color   = SERIES_COLORS[i % len(SERIES_COLORS)]
            ls_name = ls_names[i % len(ls_names)]
            rows += (
                f'<tr style="border-bottom:'
                f'1px solid #1e2a3e;">'
                f'<td style="padding:4px 8px;'
                f'color:{color};font-size:16px;">●</td>'
                f'<td style="padding:4px 8px;'
                f'color:#e0e8ff;">{i+1}</td>'
                f'<td style="padding:4px 8px;'
                f'color:#e0e8ff;">'
                f'<b>{item["label"]}</b></td>'
                f'<td style="padding:4px 8px;'
                f'color:#aaddff;">'
                f'{item["hour"]:02d}:00 UTC</td>'
                f'<td style="padding:4px 8px;'
                f'color:#88bbdd;">{ls_name}</td>'
                f'</tr>'
            )

        display(widgets.HTML(
            '<div style="background:#0d1117;'
            'border:1px solid #1e2a3e;'
            'border-radius:8px;overflow:hidden;">'
            '<table style="width:100%;'
            'border-collapse:collapse;font-size:12px;">'
            '<thead><tr style="background:#1a2233;">'
            '<th style="padding:5px 8px;color:#6699cc;">'
            'Color</th>'
            '<th style="padding:5px 8px;color:#6699cc;">'
            '#</th>'
            '<th style="padding:5px 8px;color:#6699cc;">'
            'Region</th>'
            '<th style="padding:5px 8px;color:#6699cc;">'
            'Hour (UTC)</th>'
            '<th style="padding:5px 8px;color:#6699cc;">'
            'Line Style</th>'
            '</tr></thead>'
            f'<tbody>{rows}</tbody>'
            '</table></div>'
        ))


# ============================================================
# 13) Callbacks: Country / State
# ============================================================

def on_country_change(change):
    if change['name'] != 'value':
        return
    country = change['new']
    if country == '🇺🇸 United States':
        w_state.options      = sorted(US_STATES.keys())
        w_state.value        = 'Texas'
        w_state.description  = '🗾 State:'
        w_county.description = '📍 County:'
    elif country == '🇨🇦 Canada':
        w_state.options      = CANADA_PROVINCES
        w_state.value        = CANADA_PROVINCES[0]
        w_state.description  = '🗾 Province:'
        w_county.description = '📍 District:'
    else:
        w_state.options      = MEXICO_STATES
        w_state.value        = MEXICO_STATES[0]
        w_state.description  = '🗾 Estado:'
        w_county.description = '📍 Municipio:'
    w_county.options = ['— Entire State —']
    w_county.value   = '— Entire State —'

w_country.observe(on_country_change, names='value')


def on_state_change(change):
    if change['name'] != 'value':
        return
    country = w_country.value
    state   = change['new']
    w_status.value = (
        '<span style="color:orange;font-size:13px;">'
        f'⏳ Loading sub-regions for <b>{state}</b>...</span>')
    try:
        if country == '🇺🇸 United States':
            subs = (['— Entire State —']
                    + fetch_counties_usa(state))
        elif country == '🇨🇦 Canada':
            subs = (['— Entire Province —']
                    + fetch_subdivisions(country, state))
        else:
            subs = (['— Entire State —']
                    + fetch_subdivisions(country, state))
        w_county.options = subs
        w_county.value   = subs[0]
        w_status.value   = (
            '<span style="color:green;font-size:13px;">'
            f'✅ {len(subs)-1} sub-regions loaded for '
            f'<b>{state}</b>.</span>')
    except Exception as ex:
        w_county.options = ['— Entire State —']
        w_status.value   = (
            f'<span style="color:red;font-size:13px;">'
            f'⚠️ Could not load sub-regions: {ex}</span>')

w_state.observe(on_state_change, names='value')


# ============================================================
# 14) Update Map Callback
# ============================================================

def on_update_map(b):
    country  = w_country.value
    state    = w_state.value
    county   = w_county.value
    year     = w_year.value
    month    = w_month.value
    basemap  = w_basemap.value
    mon_name = MONTH_NAMES[month - 1]

    region_label = (
        f'{county}, {state}'
        if county not in ('— Entire State —',
                          '— Entire Province —')
        else state)

    last_day   = get_days_in_month(year, month)
    start_date = f'{year}-{month:02d}-01'
    end_date   = f'{year}-{month:02d}-{last_day}'

    tz_str  = get_timezone(country, state)
    tz_abbr = get_tz_abbr(year, month, tz_str)

    # Rebuild hour dropdowns with local time labels
    hour_opts             = build_hour_options(year, month, tz_str)
    w_region_hour.options = hour_opts
    w_ts_hour.options     = hour_opts

    w_status.value = (
        '<span style="color:darkorange;font-size:13px;">'
        f'⏳ Building map — <b>{region_label}</b>  '
        f'<b>{mon_name} {year}</b>  '
        f'<i>(TZ: {tz_abbr})</i>...</span>')
    w_progress.layout.visibility = 'visible'
    w_progress.value = 0

    with sample_output:
        clear_output(wait=True)
    with ts_output:
        clear_output(wait=True)
    with region_ts_output:
        clear_output(wait=True)

    try:
        region = get_region(country, state, county)
        _current_region['fc']     = region
        _current_region['label']  = region_label
        _current_region['tz_str'] = tz_str

        Map = geemap.Map(basemap=basemap)
        Map.add_basemap(basemap)
        Map.centerObject(region, 8)
        Map.addLayer(region, {'color': 'white'},
                     f'📍 {region_label} Boundary')

        layers_added = 0
        first_hour   = None

        for hour in range(24):
            col = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
                   .filterDate(start_date, end_date)
                   .filterBounds(region)
                   .filter(ee.Filter.calendarRange(
                       hour, hour, 'hour'))
                   .select(BAND))
            size = col.size().getInfo()
            w_progress.value = hour + 1
            if size > 0:
                img        = col.mean().clip(region)
                time_label = utc_hour_to_local_label(
                    hour, year, month, tz_str)
                shown = (layers_added == 0)
                if shown:
                    first_hour = hour
                Map.addLayer(
                    img, visParams,
                    f'{mon_name} {year}  |  '
                    f'{time_label}  ({size} imgs)',
                    shown)
                layers_added += 1

        if first_hour is not None:
            w_region_hour.value = first_hour
            w_ts_hour.value     = first_hour

        # Auto-centroid for point sampler
        try:
            centroid   = (region.geometry()
                          .centroid(1).getInfo())
            clon, clat = centroid['coordinates']
            w_lat.value = round(clat, 4)
            w_lon.value = round(clon, 4)
        except Exception:
            pass

        add_colorbar_to_map(Map)

        Map.add_widget(widgets.HTML(
            '<div style="font-size:11px;'
            'background:rgba(0,0,20,0.75);'
            'color:#e0e8ff;border-radius:7px;'
            'padding:8px 12px;line-height:1.75;'
            'min-width:200px;">'
            '<b style="font-size:12px;color:#00ccff;">'
            '📊 Analysis Modes</b><br>'
            '<b style="color:#ffaa33;">🗺️ Regional:</b> '
            'Spatial mean ↓<br>'
            '<b style="color:#ff69b4;">📊 Multi-Queue:</b> '
            'Add regions/hours ↓<br>'
            '<b style="color:#00ccff;">📍 Point:</b> '
            'Exact Lat/Lon ↓<br>'
            '<span style="opacity:0.45;">'
            '──────────────────</span><br>'
            f'<b>Band:</b> {NO2_META["short_name"]}<br>'
            f'<b>Units:</b> {NO2_META["units"]}'
            '</div>'), position='bottomleft')

        with map_output:
            clear_output(wait=True)
            display(Map)

        w_progress.layout.visibility = 'hidden'
        w_status.value = (
            '<span style="color:green;font-size:13px;">'
            f'✅ <b>{layers_added}</b> hourly layers — '
            f'<b>{region_label}</b>  <b>{mon_name} {year}</b>  '
            f'🕐 <b>{tz_abbr}</b>  (<code>{tz_str}</code>).'
            '</span>')
        w_region_status.value = (
            '<span style="color:green;font-size:12px;">'
            f'✅ Map ready — <b>{region_label}</b>.  '
            'Select hour → <b>Plot Regional TS</b>  or  '
            '<b>➕ Add to Queue</b> for multi-series ↓'
            '</span>')

    except Exception as ex:
        w_progress.layout.visibility = 'hidden'
        w_status.value = (
            f'<span style="color:red;font-size:13px;">'
            f'❌ Error: {ex}</span>')

btn_update.on_click(on_update_map)


# ============================================================
# 15) Regional TS Callback (single)
# ============================================================

def on_region_ts(b):
    hour         = w_region_hour.value
    region_fc    = _current_region['fc']
    region_label = _current_region['label']
    tz_str       = _current_region['tz_str']
    year         = w_year.value
    month        = w_month.value
    local_label  = utc_hour_to_local_label(
        hour, year, month, tz_str)

    if region_fc is None:
        w_region_status.value = (
            '<span style="color:red;font-size:12px;">'
            '❌ Click <b>Update Map</b> first.</span>')
        return

    w_region_status.value = (
        '<span style="color:#ffaa00;font-size:12px;">'
        f'⏳ Computing spatial mean — '
        f'<b>{region_label}</b>  |  {local_label}  |  '
        'Apr 2023 → present...<br>'
        '<span style="font-size:11px;color:#667788;">'
        'Please wait ~30–90 s</span></span>')

    with region_ts_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="color:#ffaa00;font-size:12px;'
            'padding:10px 14px;">'
            f'⏳ Fetching regional monthly NO₂  |  '
            f'<b>{region_label}</b>  |  {local_label}  |  '
            'Apr 2023 → present...<br>'
            '<span style="font-size:11px;color:#667788;">'
            'Please wait ~30–90 s</span></div>'))

    try:
        results = ee_sample_region(region_fc, hour)
        n_valid = len([r for r in results
                       if r['value'] is not None])
        series = [{
            'label'  : f'{region_label}  |  {local_label}',
            'results': results,
            'color'  : SERIES_COLORS[0],
            'ls'     : 'solid',
        }]
        plot_multi_time_series(
            series,
            f'{region_label}  |  {local_label}',
            region_ts_output)
        w_region_status.value = (
            '<span style="color:green;font-size:12px;">'
            f'✅ Done — <b>{n_valid}</b> months  |  '
            f'{local_label}  |  <b>{region_label}</b>'
            '</span>')
    except Exception as ex:
        w_region_status.value = (
            f'<span style="color:red;font-size:12px;">'
            f'❌ Error: {ex}</span>')
        with region_ts_output:
            clear_output(wait=True)

btn_region_ts.on_click(on_region_ts)


# ============================================================
# 16) Multi-Series Queue Callbacks
# ============================================================

# ============================================================
# 16) Multi-Series Queue Callbacks
# ============================================================

def on_add_to_queue(b):
    region_fc    = _current_region['fc']
    region_label = _current_region['label']
    tz_str       = _current_region['tz_str']
    hour         = w_region_hour.value
    year         = w_year.value
    month        = w_month.value

    if region_fc is None:
        w_queue_status.value = (
            '<span style="color:red;font-size:12px;">'
            '❌ Click <b>Update Map</b> first.</span>')
        return

    local_label = utc_hour_to_local_label(
        hour, year, month, tz_str)

    # Duplicate check
    for item in _region_queue:
        if (item['label'] == region_label
                and item['hour'] == hour):
            w_queue_status.value = (
                '<span style="color:#ffaa00;font-size:12px;">'
                f'⚠️ <b>{region_label}  '
                f'{hour:02d}:00 UTC</b> already in queue.'
                '</span>')
            return

    # Capacity check
    if len(_region_queue) >= 20:
        w_queue_status.value = (
            '<span style="color:red;font-size:12px;">'
            '❌ Queue full (max 20 series).</span>')
        return

    _region_queue.append({
        'label'      : region_label,
        'region_fc'  : region_fc,
        'hour'       : hour,
        'tz_str'     : tz_str,
        'local_label': local_label,
    })
    refresh_queue_display()
    w_queue_status.value = (
        '<span style="color:green;font-size:12px;">'
        f'✅ Added: <b>{region_label}</b>  {local_label}  '
        f'({len(_region_queue)} item'
        f'{"s" if len(_region_queue) > 1 else ""} in queue)'
        '</span>')


def on_clear_queue(b):
    _region_queue.clear()
    refresh_queue_display()
    w_queue_status.value = (
        '<span style="color:gray;font-size:12px;">'
        '🗑️ Queue cleared.</span>')
    with multi_ts_output:
        clear_output(wait=True)


def on_plot_queue(b):
    if not _region_queue:
        w_queue_status.value = (
            '<span style="color:red;font-size:12px;">'
            '❌ Queue is empty.  '
            'Add regions using <b>➕ Add to Queue</b>.'
            '</span>')
        return

    n = len(_region_queue)
    w_queue_status.value = (
        '<span style="color:#ffaa00;font-size:12px;">'
        f'⏳ Fetching <b>{n} series</b>...  '
        'Please wait.</span>')

    with multi_ts_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="color:#ffaa00;font-size:12px;'
            'padding:10px 14px;">'
            f'⏳ Building {n} time series in sequence...<br>'
            '<span style="font-size:11px;color:#667788;">'
            f'Estimated wait: {n*30}–{n*90} s'
            '</span></div>'))

    series_list = []
    errors      = []

    for i, item in enumerate(_region_queue):
        w_queue_status.value = (
            '<span style="color:#ffaa00;font-size:12px;">'
            f'⏳ Fetching series {i+1}/{n}: '
            f'<b>{item["label"]}</b>  '
            f'{item["local_label"]}...'
            '</span>')
        try:
            results = ee_sample_region(
                item['region_fc'], item['hour'])
            n_valid = len([r for r in results
                           if r['value'] is not None])
            series_list.append({
                'label'  : (f'{item["label"]}  |  '
                            f'{item["local_label"]}'),
                'results': results,
                'color'  : SERIES_COLORS[
                    i % len(SERIES_COLORS)],
                'ls'     : SERIES_LINESTYLES[
                    i % len(SERIES_LINESTYLES)],
            })
        except Exception as ex:
            errors.append(
                f'{item["label"]} @ '
                f'{item["hour"]:02d}:00 UTC: {ex}')

    # Build descriptive title
    unique_regions = list(dict.fromkeys(
        [it['label'] for it in _region_queue]))
    unique_hours   = sorted(list(set(
        [it['hour'] for it in _region_queue])))
    title_extra = (
        f'{", ".join(unique_regions[:3])}'
        f'{"..." if len(unique_regions) > 3 else ""}  |  '
        f'Hours: '
        f'{", ".join([f"{h:02d}:00Z" for h in unique_hours])}'
    )

    plot_multi_time_series(
        series_list, title_extra, multi_ts_output)

    if errors:
        err_str = '<br>'.join(errors)
        w_queue_status.value = (
            '<span style="color:#ffaa00;font-size:12px;">'
            f'⚠️ {len(series_list)}/{n} series plotted.  '
            f'Errors:<br>{err_str}</span>')
    else:
        w_queue_status.value = (
            '<span style="color:green;font-size:12px;">'
            f'✅ All <b>{len(series_list)}</b> series plotted!  '
            f'Regions: '
            f'{", ".join(unique_regions[:4])}'
            f'{"..." if len(unique_regions) > 4 else ""}  |  '
            f'Hours: '
            f'{", ".join([f"{h:02d}:00Z" for h in unique_hours])}'
            '</span>')


btn_add_to_queue.on_click(on_add_to_queue)
btn_clear_queue.on_click(on_clear_queue)
btn_plot_queue.on_click(on_plot_queue)


# ============================================================
# 17) Sample & Plot Point Callback
# ============================================================

def on_sample_and_plot(b):
    lat          = w_lat.value
    lon          = w_lon.value
    hour         = w_ts_hour.value
    year         = w_year.value
    month        = w_month.value
    country      = w_country.value
    state        = w_state.value
    county       = w_county.value
    tz_str       = get_timezone(country, state)
    tz_abbr      = get_tz_abbr(year, month, tz_str)
    mon_name     = MONTH_NAMES[month - 1]
    region_label = (
        f'{county}, {state}'
        if county not in ('— Entire State —',
                          '— Entire Province —')
        else state)
    local_label  = utc_hour_to_local_label(
        hour, year, month, tz_str)

    # Validate coordinates
    if not (-90 <= lat <= 90) or not (-180 <= lon <= 180):
        w_sample_status.value = (
            '<span style="color:red;font-size:12px;">'
            '❌ Invalid coordinates.  '
            'Lat: −90→90   Lon: −180→180</span>')
        return

    # Step 1: Spinner card
    w_sample_status.value = (
        '<span style="color:#ffaa00;font-size:12px;">'
        f'⏳ Sampling at ({lat:.5f}°, {lon:.5f}°)  |  '
        f'{local_label}  |  {mon_name} {year}...</span>')

    with sample_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="background:#0d1117;color:#e0e8ff;'
            'border:1px solid #334466;border-radius:8px;'
            'padding:10px 14px;font-size:12px;'
            'line-height:1.9;">'
            f'<b style="color:#00ccff;">📍 Point:</b>  '
            f'Lat {lat:.5f}°  &nbsp;Lon {lon:.5f}°<br>'
            f'<b style="color:#00ccff;">🕐 Hour:</b>  '
            f'{local_label}<br>'
            f'<b style="color:#00ccff;">📅 Period:</b>  '
            f'{mon_name} {year}<br>'
            '<span style="color:#ffaa00;">'
            '⏳ Querying Earth Engine...</span>'
            '</div>'))

    # Step 2: Sample single value
    val = ee_sample_single(lat, lon, hour, year, month)

    with sample_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="background:#0d1117;color:#e0e8ff;'
            'border:1px solid #334466;border-radius:8px;'
            'padding:10px 14px;font-size:12px;'
            'line-height:1.9;">'
            f'<b style="font-size:13px;color:#00ccff;">'
            f'📍 Sampled Point</b><br>'
            f'<b>Lat:</b> {lat:.5f}°  &nbsp;'
            f'<b>Lon:</b> {lon:.5f}°<br>'
            f'<b>Hour:</b>  {local_label}<br>'
            f'<b>Period:</b>  {mon_name} {year}<br>'
            f'<b style="color:#00ff99;">NO₂ Value:</b>  '
            f'<b style="color:white;">{format_no2(val)}</b><br>'
            '<span style="color:#ffaa00;">'
            '⏳ Building full mission time series...</span>'
            '</div>'))

    # Step 3: Progress message
    with ts_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="color:#ffaa00;font-size:12px;'
            'padding:10px 14px;">'
            f'⏳ Fetching monthly NO₂  |  '
            f'Apr 2023 → present  |  '
            f'{local_label}  |  '
            f'({lat:.4f}°, {lon:.4f}°)...<br>'
            '<span style="font-size:11px;color:#667788;">'
            'Please wait ~30–60 s</span></div>'))

    w_sample_status.value = (
        '<span style="color:#ffaa00;font-size:12px;">'
        '⏳ Building mission-length time series — '
        'please wait...</span>')

    # Step 4: Full time series
    results = ee_sample_point(lat, lon, hour)
    n_valid = len([r for r in results
                   if r['value'] is not None])
    n_miss  = len(results) - n_valid

    series = [{
        'label'  : (f'Point ({lat:.4f}°, {lon:.4f}°)  |  '
                    f'{local_label}'),
        'results': results,
        'color'  : SERIES_COLORS[0],
        'ls'     : 'solid',
    }]
    plot_multi_time_series(
        series,
        f'Point ({lat:.4f}°, {lon:.4f}°)  |  {local_label}',
        ts_output)

    # Step 5: Final readout card
    with sample_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="background:#0d1117;color:#e0e8ff;'
            'border:1px solid #334466;border-radius:8px;'
            'padding:10px 14px;font-size:12px;'
            'line-height:1.9;">'
            f'<b style="font-size:13px;color:#00ccff;">'
            f'📍 Sampled Point</b><br>'
            f'<b>Lat:</b> {lat:.5f}°  &nbsp;'
            f'<b>Lon:</b> {lon:.5f}°<br>'
            f'<b>Hour:</b>  {local_label}<br>'
            f'<b>Period:</b>  {mon_name} {year}<br>'
            f'<b style="color:#00ff99;">NO₂ Value:</b>  '
            f'<b style="color:white;">{format_no2(val)}</b><br>'
            f'<b style="color:#aaddff;">Time Series:</b>  '
            f'<b>{n_valid}</b> months plotted  '
            f'({n_miss} missing) ↓'
            '</div>'))

    w_sample_status.value = (
        '<span style="color:green;font-size:12px;">'
        f'✅ Done!  NO₂ = <b>{format_no2(val)}</b>  |  '
        f'{n_valid} months plotted  |  '
        f'{local_label}  |  ({lat:.4f}°, {lon:.4f}°)'
        '</span>')

btn_sample.on_click(on_sample_and_plot)


# ============================================================
# 18) Full UI Assembly
# ============================================================

# ── Header ───────────────────────────────────────────────────
header = widgets.HTML("""
<div style="
    background    : linear-gradient(135deg,#001a33,#003d99);
    color         : white;
    padding       : 16px 22px;
    border-radius : 12px;
    margin-bottom : 10px;
    box-shadow    : 0 2px 12px rgba(0,100,255,0.3);">
  <h2 style="margin:0;font-size:21px;letter-spacing:0.5px;">
    🛰️ NASA TEMPO NO₂ — Interactive Explorer
  </h2>
  <p style="margin:6px 0 0 0;font-size:12.5px;
            opacity:0.85;line-height:1.8;">
    ① Select region &amp; time  →
    ② <b>Update Map</b>  →
    ③ <b>Regional TS</b> (single)  or
    ④ <b>Queue</b> multiple regions/hours →
    <b>Plot All</b>  →
    ⑤ <b>Point Sampler</b> (exact Lat/Lon)<br>
    Labels show <b>UTC + local time</b>  ·
    <b>Colorbar</b> at map bottom-right  ·
    Full mission: <b>Apr 2023 → present</b>
  </p>
</div>
""")

# ── Info banners ──────────────────────────────────────────────
info_banner = widgets.HTML(
    '<div style="display:flex;gap:8px;'
    'flex-wrap:wrap;margin:4px 0;">'

    '<div style="flex:1;min-width:160px;background:#f0f8ff;'
    'border:1px solid #b0c4de;border-radius:6px;'
    'padding:6px 10px;font-size:11px;color:#003366;">'
    '🕐 <b>Local Time:</b> All hour dropdowns show '
    'UTC + local time. DST auto-resolved.'
    '</div>'

    '<div style="flex:1;min-width:160px;background:#fff8e1;'
    'border:1px solid #ffe082;border-radius:6px;'
    'padding:6px 10px;font-size:11px;color:#5d4000;">'
    '🎨 <b>Colorbar:</b> Map bottom-right. '
    'NO₂ mol/cm².  Range: 0 → 1.5×10¹⁶.'
    '</div>'

    '<div style="flex:1;min-width:160px;background:#fff3e0;'
    'border:1px solid #ffb74d;border-radius:6px;'
    'padding:6px 10px;font-size:11px;color:#4a2000;">'
    '🗺️ <b>Regional TS:</b> Spatial mean over '
    'entire county/state.'
    '</div>'

    '<div style="flex:1;min-width:160px;background:#fce4ec;'
    'border:1px solid #f48fb1;border-radius:6px;'
    'padding:6px 10px;font-size:11px;color:#4a0020;">'
    '📊 <b>Multi-Series:</b> Queue up to 20 '
    'region + hour combos on one plot.'
    '</div>'

    '<div style="flex:1;min-width:160px;background:#f1f8e9;'
    'border:1px solid #aed581;border-radius:6px;'
    'padding:6px 10px;font-size:11px;color:#1b5e20;">'
    '📍 <b>Point Sampler:</b> Enter Lat/Lon for '
    'pixel-level value + time series.'
    '</div>'

    '</div>')

# ── Dividers ──────────────────────────────────────────────────
sep = widgets.HTML(
    '<hr style="border:1px solid #2a3a4a;margin:6px 0;">')
sep_light = widgets.HTML(
    '<hr style="border:1px dashed #2a3a4a;margin:4px 0;">')

# ── Geographic panel ──────────────────────────────────────────
geo_box = widgets.VBox([
    widgets.HTML(
        '<b style="font-size:13px;color:#003366;">'
        '📌 Geographic Selection</b>'),
    w_country, w_state, w_county,
], layout=widgets.Layout(
    padding='10px', border='1px solid #ccc',
    border_radius='8px', margin='4px'))

# ── Time panel ────────────────────────────────────────────────
time_box = widgets.VBox([
    widgets.HTML(
        '<b style="font-size:13px;color:#003366;">'
        '⏱️ Time Selection</b>'),
    w_year, w_month,
], layout=widgets.Layout(
    padding='10px', border='1px solid #ccc',
    border_radius='8px', margin='4px'))

# ── Basemap panel ─────────────────────────────────────────────
basemap_box = widgets.VBox([
    widgets.HTML(
        '<b style="font-size:13px;color:#003366;">'
        '🗺️ Basemap</b>'),
    w_basemap,
], layout=widgets.Layout(
    padding='10px', border='1px solid #ccc',
    border_radius='8px', margin='4px'))

# ── Controls row ──────────────────────────────────────────────
controls = widgets.HBox(
    [geo_box, time_box, basemap_box],
    layout=widgets.Layout(flex_flow='row wrap'))

# ── Map action bar ────────────────────────────────────────────
map_bar = widgets.VBox([
    widgets.HBox([btn_update]),
    w_progress, w_status,
], layout=widgets.Layout(padding='6px 8px'))

# ── Regional TS panel (single) ────────────────────────────────
region_ts_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient('
        '90deg,#1a0f00,#4a2800);color:white;'
        'padding:8px 14px;border-radius:8px 8px 0 0;'
        'font-size:13px;">'
        '<b>🗺️ Regional Time Series</b>'
        '<span style="font-weight:normal;font-size:11px;'
        'opacity:0.80;margin-left:8px;">'
        'Spatial mean over selected county/state  ·  '
        'Select UTC hour  ·  Apr 2023 → present'
        '</span></div>'),
    widgets.HBox(
        [w_region_hour, btn_region_ts],
        layout=widgets.Layout(
            padding='8px 10px', align_items='center',
            gap='10px', flex_flow='row wrap')),
    w_region_status,
    region_ts_output,
], layout=widgets.Layout(
    border='1px solid #7a4400',
    border_radius='8px', margin='8px 0 4px 0'))

# ── Multi-Series Queue panel ──────────────────────────────────
multi_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient('
        '90deg,#2a0030,#6a0080);color:white;'
        'padding:8px 14px;border-radius:8px 8px 0 0;'
        'font-size:13px;">'
        '<b>📊 Multi-Series Queue</b>'
        '<span style="font-weight:normal;font-size:11px;'
        'opacity:0.80;margin-left:8px;">'
        'Add up to 20 region + hour combinations  ·  '
        'overlay all on one comparison plot'
        '</span></div>'),
    widgets.HTML(
        '<div style="background:#0d0012;color:#ccaaff;'
        'font-size:11.5px;padding:7px 12px;'
        'line-height:1.7;border-bottom:1px solid #2a003a;">'
        '<b>How to use:</b>  '
        '① <b>Update Map</b> to load a region  →  '
        '② Select UTC hour above  →  '
        '③ Click <b>➕ Add Region + Hour to Queue</b>  →  '
        '④ Repeat for other regions/hours  →  '
        '⑤ Click <b>📊 Plot All Queued Series</b>'
        '</div>'),
    widgets.HBox(
        [btn_add_to_queue, btn_clear_queue, btn_plot_queue],
        layout=widgets.Layout(
            padding='8px 10px', align_items='center',
            gap='8px', flex_flow='row wrap')),
    w_queue_status,
    queue_output,
    multi_ts_output,
], layout=widgets.Layout(
    border='1px solid #6a0080',
    border_radius='8px', margin='8px 0 4px 0'))

# ── Point Sampler panel ───────────────────────────────────────
sampler_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient('
        '90deg,#001a33,#003d99);color:white;'
        'padding:8px 14px;border-radius:8px 8px 0 0;'
        'font-size:13px;">'
        '<b>📍 Point Sampler</b>'
        '<span style="font-weight:normal;font-size:11px;'
        'opacity:0.80;margin-left:8px;">'
        'Enter Lat / Lon  ·  select UTC hour  ·  '
        'Sample &amp; plot mission time series'
        '</span></div>'),
    widgets.HBox(
        [w_lat, w_lon, w_ts_hour, btn_sample],
        layout=widgets.Layout(
            flex_flow='row wrap', align_items='center',
            padding='8px 10px', gap='8px')),
    w_sample_status,
], layout=widgets.Layout(
    border='1px solid #334488',
    border_radius='8px', margin='8px 0 4px 0'))

# ── Sampled value readout panel ───────────────────────────────
sample_readout_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient('
        '90deg,#0a0a2a,#001a4d);color:white;'
        'padding:7px 14px;border-radius:8px 8px 0 0;'
        'font-size:12.5px;">'
        '<b>🔬 Sampled Point Value</b></div>'),
    sample_output,
], layout=widgets.Layout(
    border='1px solid #334466',
    border_radius='8px', margin='4px 0',
    min_height='60px'))

# ── Point time series panel ───────────────────────────────────
ts_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient('
        '90deg,#0a1a0a,#0d3300);color:white;'
        'padding:7px 14px;border-radius:8px 8px 0 0;'
        'font-size:12.5px;">'
        '<b>📈 Point Time Series</b>'
        '<span style="font-weight:normal;font-size:11px;'
        'opacity:0.75;margin-left:8px;">'
        'Monthly mean at sampled Lat/Lon  ·  '
        'Apr 2023 → present'
        '</span></div>'),
    ts_output,
], layout=widgets.Layout(
    border='1px solid #2a4a1a',
    border_radius='8px', margin='4px 0',
    min_height='80px'))

# ── Full UI ───────────────────────────────────────────────────
full_ui = widgets.VBox([
    header,
    controls,
    sep,
    info_banner,
    map_bar,
    map_output,
    sep_light,
    region_ts_panel,
    sep_light,
    multi_panel,
    sep_light,
    sampler_panel,
    sample_readout_panel,
    ts_panel,
], layout=widgets.Layout(padding='6px'))

display(full_ui)
refresh_queue_display()

# ============================================================
# 19) Auto-load Default View (Harris County, TX — Jan 2024)
# ============================================================
print("⏳ Auto-loading: Harris County, Texas — January 2024...")
w_county.options = (['— Entire State —']
                    + fetch_counties_usa('Texas'))
w_county.value   = 'Harris'
btn_update.click()

✅ Using project from Cell 4: asdc-uwg
✅ Earth Engine active — project: asdc-uwg


⏳ Auto-loading: Harris County, Texas — January 2024...
